#  VisionAI — YOLOv8 Object Detection Studio
> Run this notebook top to bottom. Each cell has instructions.

**Steps:**
1. Install dependencies
2. Start FastAPI backend with ngrok tunnel
3. Open the public URL and use the full UI

 **Enable GPU first:** Runtime → Change runtime type → T4 GPU

##  Step 1 — Install everything

In [ ]:
# Install all dependencies
# NOTE: Some packages used by Ultralytics/Matplotlib may not yet support NumPy 2.x on all platforms.
%pip install -q "numpy<2"
%pip install -q fastapi uvicorn[standard] python-multipart ultralytics pyngrok pillow nest-asyncio
print(' All packages installed!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 120.1 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.36.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 re

##  Step 2 — Paste your ngrok token
> Get a free token at https://ngrok.com → Sign up → Your Authtoken

In [ ]:
NGROK_TOKEN = "PASTE_YOUR_TOKEN_HERE"  #  replace this

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN
print(' ngrok token set!')

 ngrok token set!


##  Step 3 — Write the backend API

In [3]:
%%writefile main.py
"""
VisionAI — FastAPI + YOLOv8 backend
Runs inside Google Colab and is exposed via ngrok tunnel.
"""

from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, HTMLResponse
import uvicorn, time, uuid, os, json, base64, io
from pathlib import Path
from PIL import Image
from ultralytics import YOLO

app = FastAPI(title="VisionAI API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

Path("uploads").mkdir(exist_ok=True)
Path("results").mkdir(exist_ok=True)

print(" Loading YOLOv8x model (downloads ~136MB on first run)...")
model = YOLO("yolov8x.pt")
print(" Model ready!")


@app.get("/health")
async def health():
    return {"status": "ok", "model": "YOLOv8x", "gpu": True}


@app.get("/", response_class=HTMLResponse)
async def root():
    # Read and serve the frontend HTML
    html_path = Path("index.html")
    if html_path.exists():
        return HTMLResponse(content=html_path.read_text(), status_code=200)
    return HTMLResponse(content="<h2>VisionAI API is running! POST to /detect</h2>", status_code=200)


@app.post("/detect")
async def detect(
    file: UploadFile = File(...),
    confidence: float = 0.45,
    iou: float = 0.45,
):
    allowed = ("image/jpeg", "image/png", "image/webp", "image/jpg")
    if file.content_type not in allowed:
        raise HTTPException(400, "Only PNG/JPG/WEBP accepted")

    contents = await file.read()
    if len(contents) > 20 * 1024 * 1024:
        raise HTTPException(413, "Max 20MB")

    job_id = str(uuid.uuid4())[:8]
    img_path = f"uploads/{job_id}_{file.filename}"
    with open(img_path, "wb") as f:
        f.write(contents)

    pil_img = Image.open(io.BytesIO(contents)).convert("RGB")
    img_w, img_h = pil_img.size

    start = time.perf_counter()
    results = model.predict(source=img_path, conf=confidence, iou=iou, verbose=False)
    elapsed_ms = round((time.perf_counter() - start) * 1000, 1)

    detections = []
    for box in results[0].boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        detections.append({
            "id": len(detections),
            "class": results[0].names[int(box.cls)],
            "confidence": round(float(box.conf), 4),
            "bbox": {
                "x1": round(x1), "y1": round(y1), "x2": round(x2), "y2": round(y2),
                "width": round(x2 - x1), "height": round(y2 - y1),
                "nx": round(x1 / img_w, 4), "ny": round(y1 / img_h, 4),
                "nw": round((x2 - x1) / img_w, 4), "nh": round((y2 - y1) / img_h, 4),
            },
        })

    class_counts = {}
    for d in detections:
        class_counts[d["class"]] = class_counts.get(d["class"], 0) + 1

    avg_conf = round(sum(d["confidence"] for d in detections) / len(detections), 4) if detections else 0
    b64 = base64.b64encode(contents).decode()
    mime = file.content_type or "image/jpeg"

    response = {
        "job_id": job_id,
        "model": "YOLOv8x",
        "inference_ms": elapsed_ms,
        "image": {
            "filename": file.filename, "width": img_w, "height": img_h,
            "data_url": f"data:{mime};base64,{b64}",
        },
        "summary": {
            "total_detections": len(detections),
            "unique_classes": len(class_counts),
            "avg_confidence": avg_conf,
            "class_counts": class_counts,
        },
        "config": {"confidence_threshold": confidence, "iou_threshold": iou},
        "detections": sorted(detections, key=lambda d: d["confidence"], reverse=True),
    }

    with open(f"results/{job_id}_report.json", "w") as f:
        json.dump(response, f, indent=2)

    os.remove(img_path)
    return JSONResponse(content=response)


print("main.py written!")

Writing main.py


##  Step 4 — Upload the frontend HTML
> Run this cell — it will ask you to upload `index.html` from your computer

In [ ]:
import shutil
from pathlib import Path

# Colab-only upload helper. When running locally, just use an existing index.html.
try:
    from google.colab import files  # type: ignore
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False

if IN_COLAB:
    print(" Please upload your index.html file...")
    uploaded = files.upload()

    if 'index.html' in uploaded:
        print(' index.html uploaded successfully!')
    else:
        # Try to find any .html file
        for fname in uploaded:
            if fname.endswith('.html'):
                shutil.copy(fname, 'index.html')
                print(f' Saved {fname} as index.html')
                break
        else:
            print('  No HTML file found. The API will still work at /detect')
else:
    if Path('index.html').exists():
        print(' Found local index.html — using it.')
    else:
        print(' Not running in Colab and no index.html found; continuing without UI.')

 Please upload your index.html file...


##  Step 5 — Launch the server + get public URL
> **This is the main cell.** Run it and click the ngrok link that appears!

In [ ]:
import os
import time

# This notebook is primarily designed for Google Colab (ngrok tunnel + keep-alive).
# When running locally (e.g., executing via nbconvert), we skip the long-running server cell.
try:
    from google.colab import files  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if not IN_COLAB:
    print("Skipping server/ngrok launch (not running in Colab).")
    print("To run locally instead: `python -m uvicorn main:app --host 0.0.0.0 --port 8000 --reload`")
else:
    import nest_asyncio
    import uvicorn
    import threading
    from pyngrok import ngrok

    nest_asyncio.apply()

    # Kill any existing ngrok tunnels
    ngrok.kill()

    # Import the app
    import sys
    if 'main' in sys.modules:
        del sys.modules['main']
    import main as app_module

    # Start uvicorn in a background thread
    def run_server():
        uvicorn.run(app_module.app, host="0.0.0.0", port=8000, log_level="error")

    thread = threading.Thread(target=run_server, daemon=True)
    thread.start()
    time.sleep(2)  # wait for server to start

    # Open ngrok tunnel
    public_url = ngrok.connect(8000)

    print("\n" + "="*60)
    print(" VisionAI is LIVE!")
    print("="*60)
    print(f"\n Public URL:  {public_url}")
    print(f" API Docs:    {public_url}/docs")
    print(f"  Health:      {public_url}/health")
    print("\n Click the URL above to open the detection dashboard!")
    print("  Keep this cell running — stopping it shuts down the server.")
    print("="*60)

    # Keep alive
    try:
        while True:
            time.sleep(60)
            print("⏱ Still running... (server is alive)")
    except KeyboardInterrupt:
        ngrok.kill()
        print("\n Server stopped.")

##  Optional — Test detection directly in Colab
> Run this in a **separate cell** while the server is running above

In [ ]:
# Test with a sample image from the web
# Designed to be run when the server is actually running.
try:
    from google.colab import files  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if not IN_COLAB:
    print("Skipping API test (server is not started when running locally via nbconvert).")
else:
    import requests
    from IPython.display import Image as IPImage, display

    # Download a test image
    img_url = "https://ultralytics.com/images/bus.jpg"
    img_bytes = requests.get(img_url).content
    with open("test.jpg", "wb") as f:
        f.write(img_bytes)

    print("📷 Test image:")
    display(IPImage("test.jpg", width=400))

    # Send to our API
    res = requests.post(
        "http://localhost:8000/detect",
        files={"file": ("test.jpg", img_bytes, "image/jpeg")},
        data={"confidence": 0.45}
    )

    data = res.json()
    print(f"\n Inference time: {data['inference_ms']} ms")
    print(f" Total detections: {data['summary']['total_detections']}")
    print(f"🏷  Classes found: {data['summary']['class_counts']}")
    print("\n Top detections:")
    for d in data['detections'][:5]:
        print(f"  • {d['class']:15s} {d['confidence']*100:.1f}% conf")

##  Optional — Download all result reports

In [ ]:
import zipfile, os

# Colab-only download helper; locally we just create the zip.
try:
    from google.colab import files  # type: ignore
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False

results_dir = "results"
reports = list(os.listdir(results_dir)) if os.path.isdir(results_dir) else []

if not reports:
    print("No reports yet — run some detections first!")
else:
    with zipfile.ZipFile("all_reports.zip", "w") as zf:
        for f in reports:
            zf.write(f"{results_dir}/{f}", f)
    print(f" Zipped {len(reports)} report(s) -> all_reports.zip")
    if IN_COLAB:
        files.download("all_reports.zip")